# Robot dynamics as a scene graph

This notebook follows a real Square motion through numeric preprocessing, graph construction, five epochs of training, checkpoint reload and trajectory prediction. It uses **8 training and 2 validation episodes**, selected from the original disjoint splits. No formal test episode is evaluated. This is a pipeline demonstration, not a reproduction of the published three-seed accuracy numbers.

The initial download is about **1.62 GB** because the upstream release stores all D0 episodes in one file. Allow 4 GB free disk space. CPU is sufficient; GPU is optional. The pinned dependency set supports Python 3.10–3.12. This notebook installs no simulator: the audited decoder recovers fixtures from recorded XML and keeps world-frame states.

Use a fresh runtime and run cells in order. Output directories are protected against overwrite; choose a new `RUN` for a repeat. Colab resources and runtime duration are not guaranteed ([Colab FAQ](https://research.google.com/colaboratory/faq.html)).

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

REVISION = "a7767538deb0549b8043a7427a7792617e8bb3ce"
REPOSITORY = "https://github.com/Quinten-Wu504/RobotDynamics-CS224W-Stanford.git"
ROOT = Path(os.environ.get("ROBOT_DYNAMICS_REPO", "/content/RobotDynamics-demo")).resolve()
if not ROOT.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
    subprocess.run(["git", "checkout", "--detach", REVISION], cwd=ROOT, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
assert actual == REVISION, (actual, REVISION)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True).strip()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Source:", actual)
print("Python:", sys.version)


In [ ]:
# Set ROBOT_DYNAMICS_INSTALL=0 only when using the documented preinstalled environment.
if os.environ.get("ROBOT_DYNAMICS_INSTALL", "1") == "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/demo.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
import torch
import torch_geometric
print("PyTorch:", torch.__version__, "PyG:", torch_geometric.__version__)


## 1. Download and decode recorded motion

The downloader pins the upstream revision and checks the complete file size and SHA-256. The decoder checks the controller, 20 Hz timestamps and observation layout. It recovers fixture poses from each episode's XML rather than filling missing positions with zeros. Existing completed downloads are verified and reused; an interrupted `.partial` download must be moved aside before restarting.

In [ ]:
DATA = Path(os.environ.get("ROBOT_DYNAMICS_DATA", str(ROOT / "outputs/demo_data"))).resolve()
RUN = ROOT / "outputs/square_demo"
DATA.mkdir(parents=True, exist_ok=True)
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, check=True)
run("scripts/download_square.py", "--directory", DATA / "raw", "--tasks", "square_d0")
CACHE = DATA / "square_d0.hdf5"
if not CACHE.exists():
    run("-m", "robot_dynamics.cache_square", DATA / "raw/square_d0.hdf5", "--output", CACHE)
run("-m", "robot_dynamics.demo_square", "prepare", CACHE, "--output", RUN)
manifest = json.loads((RUN / "manifest.json").read_text())
print(manifest["purpose"])


## 2. Build the four-node graph

Nodes represent the end effector, nut, peg and table. All pairs exchange directed messages except peg–table, giving 10 edges. Node features include pose and typed robot/fixture fields; all models receive equivalent information. Current/past relative positions are inputs, not contact labels.

Each input contains four state frames and actions from the anchor up to the requested horizon. Targets are position changes after 1, 5 or 10 steps (0.05, 0.25 or 0.5 s). Normalization uses the eight training episodes only.

In [ ]:
import h5py
from robot_dynamics.normalization import fit_statistics
from robot_dynamics.learning_data import make_arrays
from robot_dynamics.graph_models import scene_nodes, scene_edges

with h5py.File(RUN / "square_demo.hdf5") as cache:
    statistics = fit_statistics(cache, manifest["d0"]["train"])
    example = make_arrays(cache, manifest["d0"]["validation"][:1], statistics)
nodes = scene_nodes(torch.from_numpy(example["state"][:1]), torch.from_numpy(example["context"][:1]))
print("Nodes [batch, history, nodes, features]:", tuple(nodes.shape))
print("Directed edges [source; destination]:", scene_edges().tolist())
print("First anchor:", int(example["anchor"][0]), "horizon:", int(example["horizon"][0]))


## 3. Train and reload

The model and training loop are the research implementation. This demonstration uses five epochs and selects the lowest error on the two validation episodes. Its selection rule and budget are deliberately separate from the formal 40-epoch experiment. It cannot support a model-comparison claim.

`best.pt` stores the selected weights; `last.pt` also supports continuation. Each checkpoint records configuration, data and source hashes. The standalone evaluation checks these identities before loading it.

In [ ]:
run("-m", "robot_dynamics.train_learning", RUN / "square_demo.hdf5", "--manifest", RUN / "manifest.json", "--config", "configs/demo_square.yaml", "--model", "temporal_gnn", "--output", RUN / "temporal_gnn")
run("-m", "robot_dynamics.evaluate_learning", RUN / "square_demo.hdf5", "--manifest", RUN / "manifest.json", "--checkpoint", RUN / "temporal_gnn/best.pt", "--output", RUN / "reload.json")
metrics = json.loads((RUN / "temporal_gnn/metrics.json").read_text())
reload = json.loads((RUN / "reload.json").read_text())
assert metrics["prediction_sha256"] == reload["prediction_sha256"]
print("Selected epoch:", metrics["best_epoch"])
print("Standalone reload: exact prediction hash match")
print("Validation metrics (demonstration only):", metrics["summary"])


## 4. Inspect the graph and predictions

The displayed episode is the first validation episode in the fixed order, not a favorable example selected by its error. Each point is a direct 0.5 s prediction conditioned on observed history and planned actions. Joining the points does not turn them into a free-running rollout or a robot control result.

In [ ]:
run("-m", "robot_dynamics.demo_square", "plot", RUN)
from IPython.display import Image, display
display(Image(filename=str(RUN / "graph.png")))
display(Image(filename=str(RUN / "trajectory.png")))
print(json.loads((RUN / "verification.json").read_text()))


## 5. Read the formal evidence separately

The archived three-seed experiments favor supervised Temporal GNN over the tested MLP/GRU controls on Square and Stack. JEPA and canonicalization do not show consistent additional gains. GRAIL is a separate history-only task: GNN beats GRU but does not beat MLP or messages disabled on the mean. All results, including these limitations, are retained.

The next cell regenerates tables and figures from the committed results without retraining. Refer to the [full report](https://github.com/Quinten-Wu504/RobotDynamics-CS224W-Stanford/blob/main/reports/stage10/README.md) and [reproduction guide](https://github.com/Quinten-Wu504/RobotDynamics-CS224W-Stanford/blob/main/docs/reproduction.md).

In [ ]:
run("-m", "robot_dynamics.final_results", "--output", RUN / "formal_figures")
display(Image(filename=str(RUN / "formal_figures/overview.png")))


## Save your run

The following archive includes the demonstration checkpoint, normalizer, metrics and figures, but excludes raw/cache datasets. Save it before the runtime ends. It is a newly trained demonstration checkpoint, not one of the formal experiment checkpoints.

In [ ]:
import zipfile
archive = ROOT / "outputs/square_demo_results.zip"
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in RUN.rglob("*"):
        if path.is_file() and path.suffix not in (".h5", ".hdf5"):
            bundle.write(path, path.relative_to(RUN))
print("Save this file:", archive)
# In Colab, use the Files sidebar to download this archive.
